## 12 — Protocolo externo ASTA: tiles e centerline sem inferência (v3)
Este notebook transforma a auditoria aprovada do ASTA em um contrato de avaliação
externo, antes da primeira predição. Ele não carrega pesos, não chama YOLO e não
executa o detector Hough nas imagens.
O ASTA fornece máscaras semânticas, não OBBs por instância. Por isso o endpoint externo
principal será uma centerline obtida por skeletonização da máscara. Essa escolha evita
inventar instâncias em cruzamentos e evita tratar segmentos desconectados como satélites
diferentes. YOLO e Hough serão comparados mais tarde depois de convertidos para
centerlines globais sob o mesmo contrato.
Execute em CPU. A validação das 178 máscaras é retomável, mostra progresso e usa
somente uma máscara local por vez. data/raw/asta_real continua somente leitura.

## 0. Ambiente, caminhos e funções de integridade
Esta seção monta o Drive e declara os hashes já aprovados. Nenhum detector é importado.

In [1]:
%pip install -q scikit-image

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image, ImageDraw
from skimage.morphology import skeletonize
import gc, hashlib, json, math, os, shutil, time, zipfile
import cv2
import numpy as np
import pandas as pd

Image.MAX_IMAGE_PIXELS=200_000_000

BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
ASTA_RAW=BACKUP_DIR/'data/raw/asta_real'
AUDIT_FINAL=BACKUP_DIR/'experiments/asta_external_v3/audit/final'
AUDIT_PREVIEWS=BACKUP_DIR/'experiments/asta_external_v3/audit/previews_checkpoint'
PROTOCOL_DRIVE=BACKUP_DIR/'experiments/asta_external_v3/protocol'
DRAFT_DRIVE=PROTOCOL_DRIVE/'draft_review'
FINAL_DRIVE=PROTOCOL_DRIVE/'final'
LOCAL=Path('/content/asta_protocol_v3')
STAGING=LOCAL/'staging_uma_mascara'
GEOM_PREVIEWS_DRIVE=PROTOCOL_DRIVE/'skeleton_previews_checkpoint'

EXPECTED_AUDIT_MANIFEST_SHA256='91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca'
EXPECTED_COMPONENTS_SHA256='cb0089cb07731e5fb233c112f315bae28207b8dad58cec0f733f829565846059'
EXPECTED_SHEETS_SHA256='2dd4a0b2b35a19bd0549c25ed65e4c01d472aafcfae88fbf24da9fd598dc40e9'
EXPECTED_INVENTORY_CONTRACT_SHA256='ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06'
EXPECTED_FINAL_MODELS_CONTRACT_SHA256='0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7'
EXPECTED_FINAL_CONFIG_SHA256='457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870'
EXPECTED_OPERATIONAL_WEIGHT_SHA256='a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c'
EXPECTED_HOUGH_CONFIG_SHA256='14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5'
RUN_ID='synthetic_v3_c41d5091de56'
CHECKPOINT_EVERY=5

for pasta in [LOCAL,STAGING,GEOM_PREVIEWS_DRIVE,PROTOCOL_DRIVE]:pasta.mkdir(parents=True,exist_ok=True)
assert ASTA_RAW.is_dir() and AUDIT_FINAL.is_dir() and AUDIT_PREVIEWS.is_dir()

def agora():return datetime.now().strftime('%H:%M:%S')
def sha256_arquivo(caminho,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for bloco in iter(lambda:arq.read(chunk),b''):h.update(bloco)
    return h.hexdigest()
def sha256_canonico(objeto):
    texto=json.dumps(objeto,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()
def verificar_hash_canonico(registro,campo,ensure_ascii=False):
    copia=dict(registro);observado=copia.pop(campo)
    texto=json.dumps(copia,sort_keys=True,separators=(',',':'),ensure_ascii=ensure_ascii)
    assert hashlib.sha256(texto.encode('utf-8')).hexdigest()==observado,f'Hash canônico divergente: {campo}'
    return observado
def gravar_json_atomico(caminho,objeto):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp')
    temporario.write_text(json.dumps(objeto,indent=2,ensure_ascii=False),encoding='utf-8')
    os.replace(temporario,caminho)
def gravar_csv_atomico(caminho,tabela):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp');tabela.to_csv(temporario,index=False)
    os.replace(temporario,caminho)

print(f'[{agora()}] Runtime CPU preparado. ASTA raw permanece somente leitura.')

Mounted at /content/drive
[12:15:26] Runtime CPU preparado. ASTA raw permanece somente leitura.


## 1. Gate dos artefatos aprovados
Confirma auditoria, CSVs, modelo operacional escolhido na validação e configuração Hough
congelada. Os pesos são apenas re-hashados no Drive; nenhum modelo é carregado.

In [2]:
hashes_auditoria=json.loads((AUDIT_FINAL/'hashes_artefatos_auditoria_asta_v3.json').read_text(encoding='utf-8'))
problemas=[]
for nome,esperado in tqdm(sorted(hashes_auditoria.items()),desc='Verificando auditoria ASTA',unit='arq'):
    caminho=AUDIT_FINAL/nome
    if not caminho.is_file() or sha256_arquivo(caminho)!=esperado:problemas.append(nome)
if problemas:raise RuntimeError(f'Artefatos da auditoria divergentes: {problemas}')

manifest_path=AUDIT_FINAL/'manifest_auditoria_asta_v3.csv'
componentes_path=AUDIT_FINAL/'componentes_mascaras_asta_v3.csv'
folhas_path=AUDIT_FINAL/'folhas_auditoria_asta_v3.zip'
resumo_auditoria=json.loads((AUDIT_FINAL/'resumo_auditoria_asta_v3.json').read_text(encoding='utf-8'))
contrato_inventario=json.loads((AUDIT_FINAL/'contrato_inventario_asta_v3.json').read_text(encoding='utf-8'))
assert sha256_arquivo(manifest_path)==EXPECTED_AUDIT_MANIFEST_SHA256
assert sha256_arquivo(componentes_path)==EXPECTED_COMPONENTS_SHA256
assert sha256_arquivo(folhas_path)==EXPECTED_SHEETS_SHA256
assert contrato_inventario['sha256_contrato_inventario']==EXPECTED_INVENTORY_CONTRACT_SHA256
assert resumo_auditoria['raw_somente_leitura'] is True
assert resumo_auditoria['yolo_executado'] is False and resumo_auditoria['hough_executado'] is False

manifest=pd.read_csv(manifest_path).sort_values('id_asta').reset_index(drop=True)
componentes=pd.read_csv(componentes_path)
assert len(manifest)==178 and manifest.id_asta.nunique()==178
assert len(componentes)==int(manifest.componentes_relevantes.sum())==291
assert manifest.sha256_imagem.nunique()==178 and manifest.sha256_mascara.nunique()==178
assert manifest.dimensoes_coincidem.astype(bool).all()
vazios=manifest.loc[manifest.mascara_pixels_positivos==0,'id_asta'].astype(str).tolist()
assert vazios==['ML1_20200328_002539_red']

final_dir=BACKUP_DIR/'experiments/yolo_final_multiseed_v3'/RUN_ID
contrato_modelos=json.loads((final_dir/'contrato_modelos_finais_v3.json').read_text(encoding='utf-8'))
config_final=json.loads((final_dir/'protocolo_treino_final_multiseed_v3.json').read_text(encoding='utf-8'))
assert verificar_hash_canonico(contrato_modelos,'sha256_contrato_modelos_finais',ensure_ascii=True)==EXPECTED_FINAL_MODELS_CONTRACT_SHA256
assert verificar_hash_canonico(config_final,'sha256_config_final',ensure_ascii=True)==EXPECTED_FINAL_CONFIG_SHA256
assert contrato_modelos['checkpoint_operacional_seed']==20260824
assert contrato_modelos['checkpoint_operacional_sha256']==EXPECTED_OPERATIONAL_WEIGHT_SHA256
operacional=next(x for x in contrato_modelos['checkpoints'] if int(x['seed'])==20260824)
peso_operacional=BACKUP_DIR/operacional['best_pt_relativo_drive']
assert peso_operacional.is_file() and sha256_arquivo(peso_operacional)==EXPECTED_OPERATIONAL_WEIGHT_SHA256

hough_config_path=BACKUP_DIR/'data/synthetic_runs'/RUN_ID/'hough_v3/config_congelada_v3.json'
hough_config=json.loads(hough_config_path.read_text(encoding='utf-8'))
assert hough_config['sha256_config']==EXPECTED_HOUGH_CONFIG_SHA256

gate_entradas={
    'status':'ENTRADAS_PROTOCOLO_ASTA_APROVADAS','pares':len(manifest),
    'componentes_relevantes':len(componentes),'mascara_vazia':vazios[0],
    'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
    'sha256_componentes':EXPECTED_COMPONENTS_SHA256,
    'sha256_contrato_modelos_finais':EXPECTED_FINAL_MODELS_CONTRACT_SHA256,
    'checkpoint_operacional_seed':20260824,
    'sha256_checkpoint_operacional':EXPECTED_OPERATIONAL_WEIGHT_SHA256,
    'sha256_hough_config':EXPECTED_HOUGH_CONFIG_SHA256,
    'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False
}
print(json.dumps(gate_entradas,indent=2,ensure_ascii=False))
print('Compartilhe este gate se qualquer hash ou contagem divergir. Caso aprovado, prossiga.')

Verificando auditoria ASTA:   0%|          | 0/5 [00:00<?, ?arq/s]

{
  "status": "ENTRADAS_PROTOCOLO_ASTA_APROVADAS",
  "pares": 178,
  "componentes_relevantes": 291,
  "mascara_vazia": "ML1_20200328_002539_red",
  "sha256_manifest_auditoria": "91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca",
  "sha256_componentes": "cb0089cb07731e5fb233c112f315bae28207b8dad58cec0f733f829565846059",
  "sha256_contrato_modelos_finais": "0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7",
  "checkpoint_operacional_seed": 20260824,
  "sha256_checkpoint_operacional": "a2b64773f32868ac952d81c33b331f1759b2250a3a04f49d5ab105f7bb79dd0c",
  "sha256_hough_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5",
  "raw_somente_leitura": true,
  "yolo_executado": false,
  "hough_executado": false
}
Compartilhe este gate se qualquer hash ou contagem divergir. Caso aprovado, prossiga.


## 2. Protocolo candidato e testes determinísticos
O grid regular usa 512 × 512, overlap de 128 e padding apenas à direita/abaixo.
Percentis são calculados no frame válido antes do recorte; padding da imagem recebe a
mediana normalizada e padding da máscara recebe zero. Predições cujo centro cair fora
do frame original são descartadas.
O endpoint primário é F1 de centerline com tolerância de 14 pixels, reportado macro por
frame positivo e pooled por comprimento. OBB-IoU não é endpoint primário no ASTA porque
a anotação oficial é uma máscara semântica.

In [3]:
def plano_eixo(tamanho,tile,stride):
    quantidade=math.ceil(max(0,tamanho-tile)/stride)+1
    inicios=[i*stride for i in range(quantidade)]
    tamanho_padded=inicios[-1]+tile
    return {'inicios':inicios,'quantidade':quantidade,'tamanho_padded':tamanho_padded,
            'padding_final':tamanho_padded-tamanho}

eixo_x=plano_eixo(10560,512,384);eixo_y=plano_eixo(10560,512,384)
assert eixo_x['quantidade']==eixo_y['quantidade']==28
assert eixo_x['padding_final']==eixo_y['padding_final']==320
cobertura=np.zeros(eixo_x['tamanho_padded'],dtype=np.uint8)
for inicio in eixo_x['inicios']:cobertura[inicio:inicio+512]=1
assert cobertura[:10560].all()

# Testes mínimos da representação oficial: vazio permanece vazio e cruzamento permanece
# uma única máscara semântica, sem tentativa de inventar duas instâncias OBB.
teste_vazio=np.zeros((64,64),dtype=bool)
teste_cruz=np.zeros((64,64),dtype=np.uint8)
cv2.line(teste_cruz,(4,4),(59,59),1,3);cv2.line(teste_cruz,(4,59),(59,4),1,3)
sk_vazio=skeletonize(teste_vazio);sk_cruz=skeletonize(teste_cruz>0)
assert int(sk_vazio.sum())==0 and int(sk_cruz.sum())>80

CONFIG_PROTOCOLO={
    'protocolo':'asta_external_protocol_candidate_v3','versao':'3.0',
    'fonte':{'record_id':'11642424','doi':'10.5281/zenodo.11642424','licenca':'cc-by-4.0',
             'pares':178,'largura':10560,'altura':10560,
             'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
             'sha256_componentes':EXPECTED_COMPONENTS_SHA256,
             'sha256_contrato_inventario':EXPECTED_INVENTORY_CONTRACT_SHA256},
    'tiles':{'tamanho':512,'overlap':128,'stride':384,'grid_x':28,'grid_y':28,
             'tiles_por_frame':784,'tiles_total':139552,'padding_direita':320,
             'padding_abaixo':320,'grid_regular':True,
             'descartar_predicao_centro_fora_frame':True},
    'preprocessamento':{'entrada':'uint8_grayscale','escopo_percentis':'frame_inteiro_valido',
             'percentil_baixo':1.0,'percentil_alto':99.8,'calculo':'histograma_exato_uint8',
             'saida':'uint8_0_255','padding_imagem':'mediana_normalizada_do_frame',
             'padding_mascara':0,'mesmo_para_yolo_e_hough':True,
             'normalizacao_fotometrica':False},
    'ground_truth':{'fonte':'mascara_semantica_oficial','representacao':'skeleton_centerline',
             'algoritmo':'skimage.morphology.skeletonize_2d','componentes_nao_sao_instancias':True,
             'mascara_vazia_id':'ML1_20200328_002539_red','mascara_vazia_papel':'negativo_real'},
    'modelo_yolo':{'familia':'yolo11n_obb','seed_operacional':20260824,
             'sha256_best_pt':EXPECTED_OPERATIONAL_WEIGHT_SHA256,'imgsz':512,
             'threshold_confianca':0.45,'iou_nms':0.7,'max_det':100,
             'obb_para_centerline':'eixo_maior_da_obb'},
    'hough':{'sha256_config_congelada':EXPECTED_HOUGH_CONFIG_SHA256,
             'recalibracao_no_asta':False},
    'merging_global':{'angulo_max_graus':5.0,'distancia_perpendicular_max_px':14.0,
             'gap_axial_max_px':128.0,'grafo_componentes_conexos':True,
             'agregacao_yolo':'ajuste_tls_ponderado_por_confianca',
             'agregacao_hough':'ajuste_tls_ponderado_por_comprimento'},
    'metricas':{'primaria':'f1_centerline_macro_frames_positivos','tolerancia_centerline_px':14.0,
             'secundarias':['precisao_recall_f1_centerline_pooled','cobertura_gt_por_frame',
                            'comprimento_fp_por_megapixel_negativo','taxa_frames_com_fp',
                            'tempo_preprocessamento','tempo_inferencia','tempo_merging','tempo_total_frame'],
             'obb_iou_primario':False,'bootstrap_frames':2000,'bootstrap_seed':20260823},
    'restricoes':{'ajuste_por_resultado_asta':False,'recalibrar_threshold':False,
             'selecionar_modelo_ou_seed':False,'yolo_executado':False,'hough_executado':False,
             'inferencia_asta_executada':False}
}
CONFIG_PROTOCOLO['sha256_protocolo_candidato']=sha256_canonico(CONFIG_PROTOCOLO)
candidato_path=LOCAL/'protocolo_asta_candidato_v3.json'
candidato_path.write_text(json.dumps(CONFIG_PROTOCOLO,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(CONFIG_PROTOCOLO,indent=2,ensure_ascii=False))
print('PROTOCOLO CANDIDATO CALCULADO, MAS AINDA NÃO PUBLICADO.')

{
  "protocolo": "asta_external_protocol_candidate_v3",
  "versao": "3.0",
  "fonte": {
    "record_id": "11642424",
    "doi": "10.5281/zenodo.11642424",
    "licenca": "cc-by-4.0",
    "pares": 178,
    "largura": 10560,
    "altura": 10560,
    "sha256_manifest_auditoria": "91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca",
    "sha256_componentes": "cb0089cb07731e5fb233c112f315bae28207b8dad58cec0f733f829565846059",
    "sha256_contrato_inventario": "ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06"
  },
  "tiles": {
    "tamanho": 512,
    "overlap": 128,
    "stride": 384,
    "grid_x": 28,
    "grid_y": 28,
    "tiles_por_frame": 784,
    "tiles_total": 139552,
    "padding_direita": 320,
    "padding_abaixo": 320,
    "grid_regular": true,
    "descartar_predicao_centro_fora_frame": true
  },
  "preprocessamento": {
    "entrada": "uint8_grayscale",
    "escopo_percentis": "frame_inteiro_valido",
    "percentil_baixo": 1.0,
    "percentil_alto": 9

## 3. Validação das centerlines oficiais, retomável
Execute primeiro com EXECUTAR_VALIDACAO_SKELETON=False. Depois do gate de entradas,
altere somente essa chave para True. A rotina copia uma máscara por vez, confere seu
SHA-256, skeletoniza, salva um preview e remove o arquivo local. O checkpoint é gravado
a cada cinco máscaras.
O resultado é um ZIP dirigido: casos com junções, alta multiplicidade, máscara vazia,
os dois exemplos críticos já conhecidos e uma amostra determinística de casos comuns.
Nenhuma revisão manual das 178 imagens individualmente é exigida.

In [6]:
EXECUTAR_VALIDACAO_SKELETON=True

CHECKPOINT_CSV=PROTOCOL_DRIVE/'checkpoint_skeleton_asta_v3.csv'
CHECKPOINT_JSON=PROTOCOL_DRIVE/'checkpoint_skeleton_asta_v3.json'
FINAL_LOCAL=LOCAL/'draft_review';FINAL_LOCAL.mkdir(parents=True,exist_ok=True)

def copiar_com_hash(origem,destino,tentativas=3,chunk=8*1024*1024):
    ultimo=None
    for tentativa in range(1,tentativas+1):
        try:
            if destino.exists():destino.unlink()
            h=hashlib.sha256()
            with origem.open('rb') as entrada,destino.open('wb') as saida:
                for bloco in iter(lambda:entrada.read(chunk),b''):
                    saida.write(bloco);h.update(bloco)
            if destino.stat().st_size!=origem.stat().st_size:raise IOError('tamanho divergente')
            return h.hexdigest()
        except (OSError,IOError) as exc:
            ultimo=exc
            if destino.exists():destino.unlink()
            if tentativa<tentativas:time.sleep(2**tentativa)
    raise IOError(f'Falha após {tentativas} tentativas: {ultimo}')

def auditar_skeleton(linha):
    origem=ASTA_RAW/linha.arquivo_mascara;local=STAGING/linha.arquivo_mascara
    hash_copia=copiar_com_hash(origem,local)
    if hash_copia!=linha.sha256_mascara:raise RuntimeError(f'Hash de máscara divergente: {linha.id_asta}')
    mask=cv2.imread(str(local),cv2.IMREAD_GRAYSCALE)
    if mask is None:raise RuntimeError(f'Máscara ilegível: {linha.id_asta}')
    if mask.shape!=(int(linha.altura_mascara),int(linha.largura_mascara)):
        raise RuntimeError(f'Dimensão divergente: {linha.id_asta}')
    binaria=mask>0;skeleton=skeletonize(binaria);sk_u8=skeleton.astype(np.uint8)
    vizinhos=cv2.filter2D(sk_u8,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk_u8.astype(np.int16)
    pixels_mask=int(binaria.sum());pixels_skeleton=int(sk_u8.sum())
    endpoints=int(((vizinhos==1)&skeleton).sum())
    junctions=int(((vizinhos>=3)&skeleton).sum())

    preview_origem=AUDIT_PREVIEWS/linha.preview_relativo
    if not preview_origem.is_file():raise RuntimeError(f'Preview da auditoria ausente: {linha.id_asta}')
    with Image.open(preview_origem) as pil:quadro=np.asarray(pil.convert('RGB')).copy()
    mini=cv2.resize(sk_u8.astype(np.float32),(512,512),interpolation=cv2.INTER_AREA)>0
    area=quadro[36:548,:512];area[mini]=np.array([0,255,255],dtype=np.uint8)
    destino_preview=GEOM_PREVIEWS_DRIVE/(linha.id_asta+'.jpg')
    Image.fromarray(quadro).save(destino_preview,quality=90,optimize=True)

    resultado={'id_asta':linha.id_asta,'arquivo_mascara':linha.arquivo_mascara,
        'sha256_mascara':linha.sha256_mascara,'mascara_pixels_positivos':pixels_mask,
        'componentes_relevantes':int(linha.componentes_relevantes),
        'skeleton_pixels':pixels_skeleton,'skeleton_endpoints_pixels':endpoints,
        'skeleton_junction_pixels':junctions,
        'skeleton_por_pixel_mascara':float(pixels_skeleton/pixels_mask) if pixels_mask else 0.0,
        'preview_skeleton':destino_preview.name}
    del mask,binaria,skeleton,sk_u8,vizinhos,quadro,area,mini;gc.collect()
    if local.exists():local.unlink()
    return resultado

if not EXECUTAR_VALIDACAO_SKELETON:
    print('Validação skeleton bloqueada. Compartilhe primeiro o gate da seção 1.')
else:
    existentes=[]
    if CHECKPOINT_JSON.is_file() and CHECKPOINT_CSV.is_file():
        contrato_cp=json.loads(CHECKPOINT_JSON.read_text(encoding='utf-8'))
        assert contrato_cp['sha256_manifest_auditoria']==EXPECTED_AUDIT_MANIFEST_SHA256
        assert contrato_cp['sha256_protocolo_candidato']==CONFIG_PROTOCOLO['sha256_protocolo_candidato']
        existentes=pd.read_csv(CHECKPOINT_CSV).to_dict('records')
    por_id={str(x['id_asta']):x for x in existentes}
    inicio=time.time()
    for indice,linha in enumerate(tqdm(manifest.itertuples(),total=len(manifest),desc='Skeleton ASTA',unit='mask'),1):
        preview_ok=(GEOM_PREVIEWS_DRIVE/(linha.id_asta+'.jpg')).is_file()
        if linha.id_asta in por_id and preview_ok:continue
        por_id[linha.id_asta]=auditar_skeleton(linha)
        if indice%CHECKPOINT_EVERY==0 or indice==len(manifest):
            tabela_cp=pd.DataFrame(por_id.values()).sort_values('id_asta')
            gravar_csv_atomico(CHECKPOINT_CSV,tabela_cp)
            gravar_json_atomico(CHECKPOINT_JSON,{
                'protocolo':'asta_skeleton_checkpoint_v3','processados':len(tabela_cp),
                'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
                'sha256_protocolo_candidato':CONFIG_PROTOCOLO['sha256_protocolo_candidato'],
                'atualizado_em_utc':datetime.now(timezone.utc).isoformat()})

    diagnostico=pd.DataFrame(por_id.values()).sort_values('id_asta').reset_index(drop=True)
    erros=[]
    if len(diagnostico)!=178 or diagnostico.id_asta.nunique()!=178:erros.append('Diagnóstico não contém 178 IDs únicos.')
    combinada=manifest[['id_asta','mascara_pixels_positivos','componentes_relevantes']].merge(
        diagnostico[['id_asta','mascara_pixels_positivos','skeleton_pixels']],
        on='id_asta',suffixes=('_manifest','_diag'))
    if not (combinada.mascara_pixels_positivos_manifest.astype(int)==combinada.mascara_pixels_positivos_diag.astype(int)).all():
        erros.append('Pixels positivos divergiram entre auditoria e validação skeleton.')
    positivos=combinada.mascara_pixels_positivos_manifest.astype(int)>0
    if not (combinada.loc[positivos,'skeleton_pixels']>0).all():erros.append('Máscara positiva sem skeleton.')
    if not (combinada.loc[~positivos,'skeleton_pixels']==0).all():erros.append('Máscara vazia gerou skeleton.')

    conhecidos=['ML1_20200328_002539_red','ML1_20220516_172703_red','ML1_20190806_172711_red']
    prioridade=[]
    prioridade+=conhecidos
    prioridade+=diagnostico.sort_values('skeleton_junction_pixels',ascending=False).loc[lambda x:x.skeleton_junction_pixels>0,'id_asta'].tolist()
    prioridade+=manifest.sort_values(['componentes_relevantes','mascara_pixels_positivos'],ascending=False).head(24).id_asta.tolist()
    rng=np.random.default_rng(20260823)
    comuns=sorted(set(manifest.id_asta.astype(str))-set(prioridade))
    amostra=rng.choice(comuns,size=min(24,len(comuns)),replace=False).tolist()
    ids_revisao=[]
    for id_asta in prioridade+amostra:
        if id_asta not in ids_revisao:ids_revisao.append(id_asta)
        if len(ids_revisao)>=80:break

    folhas=LOCAL/'folhas_revisao_skeleton';folhas.mkdir(parents=True,exist_ok=True)
    for antigo in folhas.glob('folha_*.jpg'):antigo.unlink()
    por_folha=12;colunas=4;largura_item,altura_item=384,411
    diag_idx=diagnostico.set_index('id_asta')
    for inicio_idx in range(0,len(ids_revisao),por_folha):
        lote=ids_revisao[inicio_idx:inicio_idx+por_folha];n_linhas=math.ceil(len(lote)/colunas)
        folha=Image.new('RGB',(colunas*largura_item,n_linhas*altura_item),(235,235,235))
        draw=ImageDraw.Draw(folha)
        for pos,id_asta in enumerate(lote):
            caminho=GEOM_PREVIEWS_DRIVE/(id_asta+'.jpg')
            with Image.open(caminho) as prev:
                prev=prev.convert('RGB');prev.thumbnail((largura_item,altura_item-28),Image.Resampling.LANCZOS)
                x=(pos%colunas)*largura_item+(largura_item-prev.width)//2
                y=(pos//colunas)*altura_item+28;folha.paste(prev,(x,y))
            d=diag_idx.loc[id_asta]
            titulo=f'{id_asta} | comp={int(d.componentes_relevantes)} | j={int(d.skeleton_junction_pixels)}'
            draw.text(((pos%colunas)*largura_item+5,(pos//colunas)*altura_item+5),titulo,fill='black')
        folha.save(folhas/f'folha_{inicio_idx//por_folha+1:02d}.jpg',quality=90,optimize=True)

    diagnostico_path=FINAL_LOCAL/'diagnostico_skeleton_asta_v3.csv';diagnostico.to_csv(diagnostico_path,index=False)
    zip_path=FINAL_LOCAL/'folhas_revisao_skeleton_asta_v3.zip'
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
        for folha in sorted(folhas.glob('folha_*.jpg')):zf.write(folha,arcname=folha.name)
    shutil.copy2(LOCAL/'protocolo_asta_candidato_v3.json',FINAL_LOCAL/'protocolo_asta_candidato_v3.json')
    resumo={
        'protocolo':'asta_protocol_review_v3','status':'VALIDACAO_SKELETON_PRONTA_PARA_REVISAO',
        'pares':len(diagnostico),'mascaras_positivas':int(positivos.sum()),
        'mascaras_vazias':int((~positivos).sum()),'id_mascara_vazia':'ML1_20200328_002539_red',
        'skeletons_positivos':int((diagnostico.skeleton_pixels>0).sum()),
        'frames_com_junction_pixels':int((diagnostico.skeleton_junction_pixels>0).sum()),
        'itens_revisao_dirigida':len(ids_revisao),'erros':erros,
        'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
        'sha256_protocolo_candidato':CONFIG_PROTOCOLO['sha256_protocolo_candidato'],
        'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False,
        'tempo_segundos':round(time.time()-inicio,1)}
    resumo['sha256_diagnostico_skeleton']=sha256_arquivo(diagnostico_path)
    resumo['sha256_folhas_revisao']=sha256_arquivo(zip_path)
    resumo_path=FINAL_LOCAL/'resumo_revisao_protocolo_asta_v3.json'
    resumo_path.write_text(json.dumps(resumo,indent=2,ensure_ascii=False),encoding='utf-8')
    if erros:
        print(json.dumps(resumo,indent=2,ensure_ascii=False));raise RuntimeError('Validação skeleton reprovada.')

    registro={p.name:sha256_arquivo(p) for p in sorted(FINAL_LOCAL.iterdir()) if p.is_file()}
    hashes_path=FINAL_LOCAL/'hashes_rascunho_protocolo_asta_v3.json'
    hashes_path.write_text(json.dumps(registro,indent=2),encoding='utf-8')
    DRAFT_DRIVE.mkdir(parents=True,exist_ok=True)
    for arquivo in tqdm(sorted(FINAL_LOCAL.iterdir()),desc='Publicando rascunho ASTA sem inferência',unit='arq'):
        if arquivo.is_file():shutil.copy2(arquivo,DRAFT_DRIVE/arquivo.name)
    for nome,esperado in registro.items():assert sha256_arquivo(DRAFT_DRIVE/nome)==esperado
    assert sha256_arquivo(DRAFT_DRIVE/hashes_path.name)==sha256_arquivo(hashes_path)
    print(json.dumps(resumo,indent=2,ensure_ascii=False))
    print('BACKUP_RASCUNHO_PROTOCOLO_ASTA_APROVADO_SEM_INFERENCIA')
    print('Envie o resumo e folhas_revisao_skeleton_asta_v3.zip antes de publicar o contrato.')

Skeleton ASTA:   0%|          | 0/178 [00:00<?, ?mask/s]

Publicando rascunho ASTA sem inferência:   0%|          | 0/5 [00:00<?, ?arq/s]

{
  "protocolo": "asta_protocol_review_v3",
  "status": "VALIDACAO_SKELETON_PRONTA_PARA_REVISAO",
  "pares": 178,
  "mascaras_positivas": 177,
  "mascaras_vazias": 1,
  "id_mascara_vazia": "ML1_20200328_002539_red",
  "skeletons_positivos": 177,
  "frames_com_junction_pixels": 72,
  "itens_revisao_dirigida": 80,
  "erros": [],
  "sha256_manifest_auditoria": "91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca",
  "sha256_protocolo_candidato": "33afc29c4479c0129bad8f4854f373215d588bc2566d54404e5770fc94bc02c1",
  "raw_somente_leitura": true,
  "yolo_executado": false,
  "hough_executado": false,
  "tempo_segundos": 1.5,
  "sha256_diagnostico_skeleton": "cd6a2ccce5a65fee093157c03570259b33f830aa6e6f88286390514cb7bd8388",
  "sha256_folhas_revisao": "bd98e890d6273fa3d378ac456d17fcbb016ac9f535db0046a026938af9ae50f4"
}
BACKUP_RASCUNHO_PROTOCOLO_ASTA_APROVADO_SEM_INFERENCIA
Envie o resumo e folhas_revisao_skeleton_asta_v3.zip antes de publicar o contrato.


## 4. Publicação do contrato após revisão humana
Deixe APROVAR_E_PUBLICAR_PROTOCOLO=False até compartilhar o resumo e o ZIP da seção 3.
Após a aprovação, altere somente essa chave. O contrato é criado de forma imutável; uma
versão divergente não sobrescreve a anterior. Mesmo depois deste gate, nenhuma inferência
é executada neste notebook.

In [7]:
APROVAR_E_PUBLICAR_PROTOCOLO=True

if not APROVAR_E_PUBLICAR_PROTOCOLO:
    print('Publicação bloqueada. Compartilhe primeiro o resumo e as folhas skeleton.')
else:
    resumo_path=DRAFT_DRIVE/'resumo_revisao_protocolo_asta_v3.json'
    hashes_path=DRAFT_DRIVE/'hashes_rascunho_protocolo_asta_v3.json'
    assert resumo_path.is_file() and hashes_path.is_file()
    resumo_review=json.loads(resumo_path.read_text(encoding='utf-8'))
    hashes_review=json.loads(hashes_path.read_text(encoding='utf-8'))
    assert resumo_review['status']=='VALIDACAO_SKELETON_PRONTA_PARA_REVISAO'
    assert resumo_review['erros']==[] and resumo_review['pares']==178
    assert resumo_review['sha256_protocolo_candidato']==CONFIG_PROTOCOLO['sha256_protocolo_candidato']
    for nome,esperado in hashes_review.items():assert sha256_arquivo(DRAFT_DRIVE/nome)==esperado

    contrato={
        'protocolo':'asta_external_protocol_v3','status':'APROVADO_SEM_INFERENCIA',
        'aprovado':True,'aprovado_em_utc':datetime.now(timezone.utc).isoformat(),
        'configuracao':CONFIG_PROTOCOLO,
        'sha256_manifest_auditoria':EXPECTED_AUDIT_MANIFEST_SHA256,
        'sha256_componentes':EXPECTED_COMPONENTS_SHA256,
        'sha256_diagnostico_skeleton':resumo_review['sha256_diagnostico_skeleton'],
        'sha256_folhas_revisao':resumo_review['sha256_folhas_revisao'],
        'revisao_visual_csv_aprovada':True,'validacao_skeleton_aprovada':True,
        'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False,
        'inferencia_asta_executada':False
    }
    contrato['sha256_contrato_protocolo_asta']=sha256_canonico(contrato)
    FINAL_DRIVE.mkdir(parents=True,exist_ok=True)
    contrato_path=FINAL_DRIVE/'contrato_protocolo_asta_v3.json'
    if contrato_path.exists():
        existente=json.loads(contrato_path.read_text(encoding='utf-8'))
        if existente.get('sha256_contrato_protocolo_asta')!=contrato['sha256_contrato_protocolo_asta']:
            raise RuntimeError('Já existe contrato ASTA diferente. Não sobrescrever.')
        contrato=existente
    else:
        with contrato_path.open('x',encoding='utf-8') as arq:json.dump(contrato,arq,indent=2,ensure_ascii=False)
    for nome in ['resumo_revisao_protocolo_asta_v3.json','diagnostico_skeleton_asta_v3.csv',
                 'folhas_revisao_skeleton_asta_v3.zip','protocolo_asta_candidato_v3.json']:
        origem=DRAFT_DRIVE/nome;destino=FINAL_DRIVE/nome
        if not destino.exists():shutil.copy2(origem,destino)
        assert sha256_arquivo(destino)==sha256_arquivo(origem)
    ponteiro={'protocolo':'asta_external_protocol_v3',
              'sha256_contrato_protocolo_asta':contrato['sha256_contrato_protocolo_asta'],
              'contrato_relativo':'experiments/asta_external_v3/protocol/final/contrato_protocolo_asta_v3.json',
              'inferencia_asta_executada':False}
    gravar_json_atomico(PROTOCOL_DRIVE/'LATEST_ASTA_PROTOCOL_V3.json',ponteiro)
    print(json.dumps(contrato,indent=2,ensure_ascii=False))
    print('PROTOCOLO_ASTA_V3_PUBLICADO_SEM_INFERENCIA')

{
  "protocolo": "asta_external_protocol_v3",
  "status": "APROVADO_SEM_INFERENCIA",
  "aprovado": true,
  "aprovado_em_utc": "2026-08-30T12:48:33.371538+00:00",
  "configuracao": {
    "protocolo": "asta_external_protocol_candidate_v3",
    "versao": "3.0",
    "fonte": {
      "record_id": "11642424",
      "doi": "10.5281/zenodo.11642424",
      "licenca": "cc-by-4.0",
      "pares": 178,
      "largura": 10560,
      "altura": 10560,
      "sha256_manifest_auditoria": "91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca",
      "sha256_componentes": "cb0089cb07731e5fb233c112f315bae28207b8dad58cec0f733f829565846059",
      "sha256_contrato_inventario": "ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06"
    },
    "tiles": {
      "tamanho": 512,
      "overlap": 128,
      "stride": 384,
      "grid_x": 28,
      "grid_y": 28,
      "tiles_por_frame": 784,
      "tiles_total": 139552,
      "padding_direita": 320,
      "padding_abaixo": 320,
      "grid

## 5. Verificação pós-backup
Este gate pode ser executado após reconexão. Ele apenas verifica o contrato publicado.

In [8]:
contrato_path=FINAL_DRIVE/'contrato_protocolo_asta_v3.json'
if not contrato_path.is_file():
    print('Contrato final ainda não foi publicado.')
else:
    contrato=json.loads(contrato_path.read_text(encoding='utf-8'))
    assert verificar_hash_canonico(contrato,'sha256_contrato_protocolo_asta')==contrato['sha256_contrato_protocolo_asta']
    assert contrato['status']=='APROVADO_SEM_INFERENCIA' and contrato['aprovado'] is True
    assert contrato['raw_somente_leitura'] is True
    assert contrato['yolo_executado'] is False and contrato['hough_executado'] is False
    assert contrato['inferencia_asta_executada'] is False
    print(json.dumps({
        'status':'BACKUP_PROTOCOLO_ASTA_V3_APROVADO_SEM_INFERENCIA',
        'sha256_contrato_protocolo_asta':contrato['sha256_contrato_protocolo_asta'],
        'sha256_manifest_auditoria':contrato['sha256_manifest_auditoria'],
        'sha256_diagnostico_skeleton':contrato['sha256_diagnostico_skeleton'],
        'yolo_executado':False,'hough_executado':False
    },indent=2,ensure_ascii=False))

{
  "status": "BACKUP_PROTOCOLO_ASTA_V3_APROVADO_SEM_INFERENCIA",
  "sha256_contrato_protocolo_asta": "dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc",
  "sha256_manifest_auditoria": "91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca",
  "sha256_diagnostico_skeleton": "cd6a2ccce5a65fee093157c03570259b33f830aa6e6f88286390514cb7bd8388",
  "yolo_executado": false,
  "hough_executado": false
}
